# ref 

https://sbert.net/docs/sentence_transformer/pretrained_models.html#semantic-search-models

In [5]:
# pip install -U sentence-transformers
# pip install torch

In [1]:
from sentence_transformers import SentenceTransformer
import torch

/Users/peetiphartsuebpeng/Documents/CONDA_ENV/env-wizmap/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# Class 

class SementicSearch :
    def __init__(self, document, model_name='paraphrase-multilingual-mpnet-base-v2', verbose=True, device=None):
        self.device = device if device is not None else 'cpu'
        self.model_name = model_name
        self.verbose = verbose
        self.sbert = SentenceTransformer(self.model_name, device=self.device)
        self.corpus = document
        self.corpus_embeddings = self.sbert.encode(
            self.corpus, 
            convert_to_tensor=True, 
            batch_size=126, 
            show_progress_bar=self.verbose
        )

    def search(self, query, top_k=5):
        query_results = []
        query_embedding = self.sbert.encode(query, convert_to_tensor=True)
        similarity_scores = self.sbert.similarity(query_embedding, self.corpus_embeddings)[0]
        scores, indices = torch.topk(similarity_scores, k=top_k)
        for score, idx in zip(scores, indices):
            query_results.append(
                {
                    'idx': idx.item(),
                    'query': self.corpus[idx], 
                    'score': score.item()
                }
            )
        return query_results

In [3]:
documents = [
    "The cat sat on the mat.",
    "The dog barked loudly.",
    "The quick brown fox jumps over the lazy dog.",
    "A fox is a quick, cunning animal.",
    "Dogs are loyal and intelligent pets."
]

In [4]:
class_sementic = SementicSearch(
    document=documents, 
    model_name='all-mpnet-base-v2', 
    verbose=True
)

/Users/peetiphartsuebpeng/Documents/CONDA_ENV/env-wizmap/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
query = 'dog very lazy'

In [7]:
query_result = class_sementic.search(query=query, top_k=1)

In [8]:
query_result

[{'idx': 2,
  'query': 'The quick brown fox jumps over the lazy dog.',
  'score': 0.37146836519241333}]

In [9]:
query_result = class_sementic.search(query=query, top_k=3)
query_result

[{'idx': 2,
  'query': 'The quick brown fox jumps over the lazy dog.',
  'score': 0.37146836519241333},
 {'idx': 4,
  'query': 'Dogs are loyal and intelligent pets.',
  'score': 0.3493441045284271},
 {'idx': 1, 'query': 'The dog barked loudly.', 'score': 0.2746102213859558}]